# Prediksi Return USD/IDR dari Berita Geopolitik (NLP)

Pipeline: cleaning berita & kurs BI → skor sentimen (VADER pada judul, Loughran-McDonald pada isi) → alignment ke hari bursa → agregasi harian → regresi return.

In [1]:
import os
import re
import html
import unicodedata
import warnings
from collections import Counter
from datetime import time

import numpy as np
import pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.corpus import wordnet
from nltk import pos_tag
from nltk.stem import WordNetLemmatizer
from langdetect import detect, DetectorFactory, LangDetectException

DetectorFactory.seed = 42
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

# Download resource NLTK yang dibutuhkan (butuh koneksi internet sekali saja)
for pkg in ["stopwords", "wordnet", "omw-1.4", "averaged_perceptron_tagger_eng"]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print(f"Gagal mengunduh '{pkg}': {e}")


In [2]:
news_raw = pd.read_csv('data/raw/cnbc_with_published.csv')
bi_raw = pd.read_csv('data/raw/bi-usd-rate.csv')

print("Berita geopolitik :", news_raw.shape)
print("Kurs USD BI       :", bi_raw.shape)
news_raw.head(5)


Berita geopolitik : (16733, 13)
Kurs USD BI       : (1202, 5)


,keyword_matched,title,url,preview_summary,full_text,date_raw,section,published_raw,published_date,published_time,published_timezone,published_iso,scrape_status
0,geopolitical risk,Global shipping authorities warn of maritime trade breakdown amidgeopoliticalturmoil,https://www.cnbc.com/2026/09/08/global-shipping-iran-war-hormuz-rules.html?&qsearchterm=geopolitical risk,Global maritime authorities have warned that the emergence of “parallel systems” threatens to create a two-tier stru...,"Global maritime authorities have warned that the emergence of ""parallel systems"" threatens to create a two-tier stru...",9/8/2026 2:28:41 PM,Markets,2026-09-08T07:28:41+0000,2026-09-08,07:28 AM,0.0,2026-09-08T07:28:41+0000,ok
1,geopolitical risk,Morning Call Sheet: AI rebound meets risinggeopoliticalrisks,https://www.cnbc.com/video/2026/07/31/morning-call-sheet-ai-rebound-meets-rising-geopolitical-risks.html?&qsearchter...,"Peter Tchir, Head of Macro Strategy at Academy Securities, Thomas Martin, Senior Portfolio Manager at GLOBALT Invest...",NaN,7/31/2026 5:58:53 PM,Morning Call,NaN,NaN,NaN,NaN,NaN,not_found
2,geopolitical risk,"Global markets keep shrugging off shocks. Here’s what could break that streak, according to HSBC",https://www.cnbc.com/2026/09/08/global-markets-shrug-off-shocks-hsbc-sees-what-could-break-the-streak.html?&qsearcht...,"Global markets have shrugged off a barrage of shocks in recent years, but HSBC sees several developments that could ...","In this article Global markets have shrugged off a barrage of shocks in recent years, but HSBC sees several developm...",9/8/2026 11:02:43 AM,World Markets,2026-09-08T04:02:43+0000,2026-09-08,04:02 AM,0.0,2026-09-08T04:02:43+0000,ok
3,geopolitical risk,"Yen hovers near seven-month high, dollar steadies",https://www.cnbc.com/2026/09/08/yen-extends-rally-to-new-seven-month-high-dollar-subdued-ahead-of-cpi.html?&qsearcht...,"The Japanese yen hovered near ​a seven-month high on Tuesday, while the dollar was little changed against major peer...","In this article The Japanese yen hovered near ​a seven-month high on Tuesday, while the dollar was little changed ag...",9/8/2026 11:17:07 AM,Currencies,2026-09-08T04:17:07+0000,2026-09-08,04:17 AM,0.0,2026-09-08T04:17:07+0000,ok
4,geopolitical risk,Central banks are bringing gold reserves home asgeopoliticalrisks rise,https://www.cnbc.com/2026/06/17/central-banks-gold-reserves-domestic-storage.html?&qsearchterm=geopolitical risk,"More and more central banks are storing gold bullion at home rather than overseas, as they expect to buy more of the...","In this article More and more central banks are storing gold bullion at home rather than overseas, as they expect to...",6/17/2026 2:28:36 PM,Gold,2026-06-17T07:28:36+0000,2026-06-17,07:28 AM,0.0,2026-06-17T07:28:36+0000,ok


In [3]:
news_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16733 entries, 0 to 16732
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   keyword_matched     16733 non-null  object 
 1   title               16733 non-null  object 
 2   url                 16733 non-null  object 
 3   preview_summary     16733 non-null  object 
 4   full_text           13431 non-null  object 
 5   date_raw            16733 non-null  object 
 6   section             16649 non-null  object 
 7   published_raw       15239 non-null  object 
 8   published_date      15239 non-null  object 
 9   published_time      15239 non-null  object 
 10  published_timezone  15239 non-null  float64
 11  published_iso       15239 non-null  object 
 12  scrape_status       16733 non-null  object 
dtypes: float64(1), object(12)
memory usage: 1.7+ MB


In [4]:
# Parse timestamp mentah GEO dan pisahkan tanggal & jam dengan timezone WIB.
news_raw["date_raw_dt"] = pd.to_datetime(news_raw["date_raw"], errors="coerce")
news_raw["date_raw_dt_wib"] = (
    pd.to_datetime(news_raw["date_raw_dt"], errors="coerce", utc=True)
    .dt.tz_convert("Asia/Jakarta")
    .dt.tz_localize(None)
)
news_raw["tanggal"] = news_raw["date_raw_dt_wib"].dt.strftime("%m/%d/%Y")
news_raw["jam"] = news_raw["date_raw_dt_wib"].dt.strftime("%I:%M %p")
news_raw = news_raw.drop(columns=["date_raw"], errors="ignore")

print("Kolom Berita Geopolitik:")
news_raw[["title", "tanggal", "jam", "date_raw_dt_wib"]].head(5)


Kolom Berita Geopolitik:


,title,tanggal,jam,date_raw_dt_wib
0,Global shipping authorities warn of maritime trade breakdown amidgeopoliticalturmoil,09/08/2026,09:28 PM,2026-09-08 21:28:41
1,Morning Call Sheet: AI rebound meets risinggeopoliticalrisks,08/01/2026,12:58 AM,2026-08-01 00:58:53
2,"Global markets keep shrugging off shocks. Here’s what could break that streak, according to HSBC",09/08/2026,06:02 PM,2026-09-08 18:02:43
3,"Yen hovers near seven-month high, dollar steadies",09/08/2026,06:17 PM,2026-09-08 18:17:07
4,Central banks are bringing gold reserves home asgeopoliticalrisks rise,06/17/2026,09:28 PM,2026-06-17 21:28:36


In [5]:
bi_raw["Tanggal_dt"] = pd.to_datetime(bi_raw["Tanggal"], errors="coerce")
bi_raw["tanggal"] = bi_raw["Tanggal_dt"].dt.strftime("%m/%d/%Y")
bi_raw["jam"] = bi_raw["Tanggal_dt"].dt.strftime("%I:%M %p")
bi_raw = bi_raw.drop(
    columns=["Tanggal"],
    errors="ignore"
)

print("Kolom Kurs BI:")
bi_raw.head(5)

Kolom Kurs BI:


,NO,Nilai,Kurs Jual,Kurs Beli,Tanggal_dt,tanggal,jam
0,1,1,17834.73,17657.27,2026-09-01,09/01/2026,12:00 AM
1,2,1,17791.51,17614.49,2026-08-31,08/31/2026,12:00 AM
2,3,1,17850.81,17673.19,2026-08-28,08/28/2026,12:00 AM
3,4,1,17805.58,17628.42,2026-08-27,08/27/2026,12:00 AM
4,5,1,17791.51,17614.49,2026-08-26,08/26/2026,12:00 AM


# Data Cleaning
## Filter Dataset

In [6]:
#delete duplicate rows
def word_count(text):
    if not isinstance(text, str):
        return 0
    return len(text.split())

news = news_raw.copy()
# Ukuran data awal
print(f"Baris awal                         : {len(news_raw)}")

# Filter rentang tanggal inklusif: 1 September 2021 - 1 September 2026
start_date = pd.Timestamp("2021-09-01")
end_date = pd.Timestamp("2026-09-01 23:59:59")
news = news[news["date_raw_dt"].between(start_date, end_date, inclusive="both")]
print(f"Setelah filter rentang 1 September 2021 - 1 September 2026 : {len(news)}")

# Setelah di drop URL
news = news.drop_duplicates(subset=["url"], keep="first")
print(f"Setelah dedup url                  : {len(news)}")

# Setelah di drop title + date_raw
news = news.drop_duplicates(subset=["title", "tanggal", "jam"], keep="first")
print(f"Setelah dedup title+date_raw       : {len(news)}")

Baris awal                         : 16733
Setelah filter rentang 1 September 2021 - 1 September 2026 : 8871
Setelah dedup url                  : 8871
Setelah dedup title+date_raw       : 7084


In [7]:
# Filter full_text yang kosong

# Buang baris yang tidak memiliki isi teks pada ketiga sumber konten.
text_columns = ["title", "preview_summary", "full_text"]
text_content = news[text_columns].fillna("").astype(str).agg(" ".join, axis=1).str.strip()
mask_empty = text_content.eq("")
news = news.loc[~mask_empty].copy()
print(f"Setelah drop teks kosong             : {len(news)}")

# Buang item multimedia berdasarkan URL, judul, dan section.
non_article_pattern = r"\b(video|videos|image|images|photo|photos|foto|gambar|audio|podcast|gallery|galeri|live)\b|\.(mp4|mov|avi|mp3|wav|jpg|jpeg|png)(?:$|[?#])"
content_metadata = news[["url", "title", "section"]].fillna("").astype(str).agg(" ".join, axis=1)
mask_non_article = content_metadata.str.contains(non_article_pattern, case=False, regex=True, na=False)
news = news.loc[~mask_non_article].copy()
print(f"Setelah drop video/gambar/audio      : {len(news)}")

news.head(5)

Setelah drop teks kosong             : 7084
Setelah drop video/gambar/audio      : 6214


,keyword_matched,title,url,preview_summary,full_text,section,published_raw,published_date,published_time,published_timezone,published_iso,scrape_status,date_raw_dt,date_raw_dt_wib,tanggal,jam
4,geopolitical risk,Central banks are bringing gold reserves home asgeopoliticalrisks rise,https://www.cnbc.com/2026/06/17/central-banks-gold-reserves-domestic-storage.html?&qsearchterm=geopolitical risk,"More and more central banks are storing gold bullion at home rather than overseas, as they expect to buy more of the...","In this article More and more central banks are storing gold bullion at home rather than overseas, as they expect to...",Gold,2026-06-17T07:28:36+0000,2026-06-17,07:28 AM,0.0,2026-06-17T07:28:36+0000,ok,2026-06-17 14:28:36,2026-06-17 21:28:36,06/17/2026,09:28 PM
8,geopolitical risk,Oil rises 2% as White House says no US-Iran talks happening,https://www.cnbc.com/2026/08/27/oil-prices-extend-losses-on-expectations-talks-to-ease-middle-east-supply-woes.html?...,"Oil prices rose Thursday, after Washington confirmed it was not in talks with ‌Iran despite diplomatic efforts by ot...","In this article Oil prices rose Thursday, after Washington confirmed it was not in talks with ‌Iran despite diplomat...",Oil,2026-08-27T02:49:03+0000,2026-08-27,02:49 AM,0.0,2026-08-27T02:49:03+0000,ok,2026-08-27 09:49:03,2026-08-27 16:49:03,08/27/2026,04:49 PM
9,geopolitical risk,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ...,https://www.cnbc.com/2026/07/21/jpmorgan-chase-ceo-jamie-dimon-market-risk.html?&qsearchterm=geopolitical risk,JPMorgan Chase CEO Jamie Dimon said investors are underestimating the risks facing the global economy and that he wo...,In this article JPMorgan ChaseCEOJamie Dimonsaid investors are underestimating the risks facing the global economy a...,Finance,2026-07-20T23:01:01+0000,2026-07-20,11:01 PM,0.0,2026-07-20T23:01:01+0000,ok,2026-07-21 06:01:01,2026-07-21 13:01:01,07/21/2026,01:01 PM
13,geopolitical risk,"Citi Wealth warns markets may be ‘uncomfortably strong’ amid mountinggeopolitical, inflation risks",https://www.cnbc.com/2026/05/18/citi-wealth-warns-markets-may-be-uncomfortably-strong-as-risks-mount.html?&qsearchte...,"Global markets may be due for a period of consolidation after a sharp rally in equities, even as the longer-term out...",NaN,Pro: Analysis,2026-05-18T05:37:18+0000,2026-05-18,05:37 AM,0.0,2026-05-18T05:37:18+0000,ok,2026-05-18 12:37:18,2026-05-18 19:37:18,05/18/2026,07:37 PM
15,geopolitical risk,Where fixed income investors are finding yield asgeopoliticalriskrattles markets,https://www.cnbc.com/2026/04/10/where-fixed-income-investors-are-finding-yield-as-geopolitical-risk-rattles-markets....,"As the Iran war shakes up markets, strategists say there are still plenty of sources of relatively safe yield for in...",NaN,Pro: Income Investing,2026-04-10T19:25:17+0000,2026-04-10,07:25 PM,0.0,2026-04-10T19:25:17+0000,ok,2026-04-11 02:25:17,2026-04-11 09:25:17,04/11/2026,09:25 AM


In [8]:
# Panjang text minimal 350

effective_text = news["full_text"].fillna(news["preview_summary"])
news = news.loc[effective_text.apply(word_count) >= 350]
print(f"Setelah filter panjang teks min     : {len(news)}")

Setelah filter panjang teks min     : 4349


In [9]:
#Filter keyword yang relevan dengan geopolitik dan ekonomi
RELEVANCE_KEYWORDS = [
    "geopolitics",
    "geopolitical risk",
    "geopolitical tensions",
    "geopolitical fragmentation",
    "armed conflict",
    "global conflict",
    "international conflict",
    "military tensions",
    "national security",
    "sanctions",
    "trade war",
    "tariffs",
    "embargo",
    "export controls",
    "protectionism",
    "supply chain disruption",
    "diplomacy",
    "foreign policy",
    "nato",
    "brics",
    "opec",
    "g7",
]

EXCLUDE_SECTIONS = [
    "sports", "entertainment", "television", "lifestyle", "travel",
    "food retail", "beer, wine & spirits", "college", "modern medicine",
    "life changes", "fashion", "celebrity",
]

def is_relevant(row):
    """Berita dianggap relevan jika keyword_matched terisi ATAU teks memuat
    salah satu kata kunci geopolitik/ekonomi pada RELEVANCE_KEYWORDS."""
    if isinstance(row["keyword_matched"], str) and row["keyword_matched"].strip():
        return True
    haystack = f"{row.get('title', '')} {row.get('preview_summary', '')}".lower()
    return any(kw in haystack for kw in RELEVANCE_KEYWORDS)

mask_relevant = news.apply(is_relevant, axis=1)
news = news.loc[mask_relevant]

print(f"Setelah filter relevansi kata kunci : {len(news)}")

# 4.5 Filter kategori/section yang tidak relevan
section_lower = news["section"].fillna("").str.lower()
mask_section = ~section_lower.isin(EXCLUDE_SECTIONS)
news = news.loc[mask_section].reset_index(drop=True)

print(f"Setelah filter kategori/section     : {len(news)}")

news.head(3)


Setelah filter relevansi kata kunci : 4349
Setelah filter kategori/section     : 4336


,keyword_matched,title,url,preview_summary,full_text,section,published_raw,published_date,published_time,published_timezone,published_iso,scrape_status,date_raw_dt,date_raw_dt_wib,tanggal,jam
0,geopolitical risk,Central banks are bringing gold reserves home asgeopoliticalrisks rise,https://www.cnbc.com/2026/06/17/central-banks-gold-reserves-domestic-storage.html?&qsearchterm=geopolitical risk,"More and more central banks are storing gold bullion at home rather than overseas, as they expect to buy more of the...","In this article More and more central banks are storing gold bullion at home rather than overseas, as they expect to...",Gold,2026-06-17T07:28:36+0000,2026-06-17,07:28 AM,0.0,2026-06-17T07:28:36+0000,ok,2026-06-17 14:28:36,2026-06-17 21:28:36,06/17/2026,09:28 PM
1,geopolitical risk,Oil rises 2% as White House says no US-Iran talks happening,https://www.cnbc.com/2026/08/27/oil-prices-extend-losses-on-expectations-talks-to-ease-middle-east-supply-woes.html?...,"Oil prices rose Thursday, after Washington confirmed it was not in talks with ‌Iran despite diplomatic efforts by ot...","In this article Oil prices rose Thursday, after Washington confirmed it was not in talks with ‌Iran despite diplomat...",Oil,2026-08-27T02:49:03+0000,2026-08-27,02:49 AM,0.0,2026-08-27T02:49:03+0000,ok,2026-08-27 09:49:03,2026-08-27 16:49:03,08/27/2026,04:49 PM
2,geopolitical risk,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ...,https://www.cnbc.com/2026/07/21/jpmorgan-chase-ceo-jamie-dimon-market-risk.html?&qsearchterm=geopolitical risk,JPMorgan Chase CEO Jamie Dimon said investors are underestimating the risks facing the global economy and that he wo...,In this article JPMorgan ChaseCEOJamie Dimonsaid investors are underestimating the risks facing the global economy a...,Finance,2026-07-20T23:01:01+0000,2026-07-20,11:01 PM,0.0,2026-07-20T23:01:01+0000,ok,2026-07-21 06:01:01,2026-07-21 13:01:01,07/21/2026,01:01 PM


In [10]:
# delete missing values
jumlah_nan = news["full_text"].isna().sum()

print("Jumlah full_text NaN:", jumlah_nan)

news_nan = news.loc[news["full_text"].isna()]

news_nan[
    ["url", "title", "preview_summary", "full_text"]
].head()

news = news.dropna(subset=["full_text"]).reset_index(drop=True)

print("Jumlah baris setelah drop NaN:", len(news))
print("Sisa NaN pada full_text:", news["full_text"].isna().sum())

Jumlah full_text NaN: 0
Jumlah baris setelah drop NaN: 4336
Sisa NaN pada full_text: 0


## 2. Data News Cleaning

In [11]:
#menggabungkan title, preview, dan full_text

# Tidak ada nilai kosong yang mengganggu penggabungan teks.
news[["title", "preview_summary", "full_text"]] = (
    news[["title", "preview_summary", "full_text"]].fillna("")
)

# Gabungkan title + preview_summary + full_text.
news["combined_raw"] = (
    ((news["title"] + " ") * 2)
    + news["preview_summary"] + " "
    + news["full_text"]
)

print(f"Jumlah baris: {len(news)}")
news[["title", "combined_raw"]].head(3)

Jumlah baris: 4336


,title,combined_raw
0,Central banks are bringing gold reserves home asgeopoliticalrisks rise,Central banks are bringing gold reserves home asgeopoliticalrisks rise Central banks are bringing gold reserves home...
1,Oil rises 2% as White House says no US-Iran talks happening,Oil rises 2% as White House says no US-Iran talks happening Oil rises 2% as White House says no US-Iran talks happen...
2,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ...,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ... Jamie Dimon says...


In [12]:
import unicodedata
from collections import Counter

symbol_counter = Counter()
for text in news["combined_raw"]:
    for ch in text:
        if not (ch.isalnum() or ch.isspace()):
            symbol_counter[ch] += 1

symbol_df = pd.DataFrame(
    [
        (ch, unicodedata.name(ch, "UNKNOWN"), count)
        for ch, count in symbol_counter.most_common()
    ],
    columns=["simbol", "nama_unicode", "jumlah"],
)

print(f"Jumlah jenis simbol unik pada combined_raw: {len(symbol_df)}")
symbol_df

Jumlah jenis simbol unik pada combined_raw: 54


,simbol,nama_unicode,jumlah
0,.,FULL STOP,232989
1,",",COMMA,203423
2,"""",QUOTATION MARK,88912
3,',APOSTROPHE,60351
4,-,HYPHEN-MINUS,41818
5,%,PERCENT SIGN,18915
6,—,EM DASH,10379
7,$,DOLLAR SIGN,9806
8,’,RIGHT SINGLE QUOTATION MARK,6782
9,:,COLON,5100


In [13]:
#cleaning dari tanda baca yang tidak berguna

URL_RE = re.compile(r"http\S+|www\.\S+")
EMAIL_RE = re.compile(r"\S+@\S+\.\S+")
HTML_TAG_RE = re.compile(r"<[^>]+>")
EMOJI_RE = re.compile(
    "["
    "\U0001F300-\U0001FAFF"
    "\U00002700-\U000027BF"
    "\U0001F1E6-\U0001F1FF"
    "\U00002600-\U000026FF"
    "]+",
    flags=re.UNICODE,
)

# Hanya simbol mata uang yang diubah jadi kata.
# Simbol lain (&, +, tanda baca umum) dibuang di NON_ALNUM_RE.
CURRENCY_WORDS = {
    "%": "percent",
    "$": "dollar",
    "£": "pound",
    "€": "euro",
    "¥": "yen",
}
CURRENCY_RE = re.compile("|".join(re.escape(sym) for sym in CURRENCY_WORDS))
NON_ALNUM_RE = re.compile(r"[^a-zA-Z0-9\s]")  # simbol/tanda baca sisa dibuang; huruf & angka dipertahankan
MULTI_SPACE_RE = re.compile(r"\s+")


def normalize_text(text: str) -> str:
    """Hapus HTML/URL/email/emoji, ubah simbol mata uang jadi kata, angka dipertahankan."""
    if not isinstance(text, str) or not text.strip():
        return ""
    t = html.unescape(text)
    t = HTML_TAG_RE.sub(" ", t)
    t = URL_RE.sub(" ", t)
    t = EMAIL_RE.sub(" ", t)
    t = EMOJI_RE.sub(" ", t)
    t = CURRENCY_RE.sub(lambda m: f" {CURRENCY_WORDS[m.group(0)]} ", t)
    t = NON_ALNUM_RE.sub(" ", t)
    t = MULTI_SPACE_RE.sub(" ", t).strip()
    return t


news["normalized_text"] = news["combined_raw"].apply(normalize_text)

news[["combined_raw", "normalized_text"]].head(3)

,combined_raw,normalized_text
0,Central banks are bringing gold reserves home asgeopoliticalrisks rise Central banks are bringing gold reserves home...,Central banks are bringing gold reserves home asgeopoliticalrisks rise Central banks are bringing gold reserves home...
1,Oil rises 2% as White House says no US-Iran talks happening Oil rises 2% as White House says no US-Iran talks happen...,Oil rises 2 percent as White House says no US Iran talks happening Oil rises 2 percent as White House says no US Ira...
2,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ... Jamie Dimon says...,Jamie Dimon says markets underestimate risks and he wouldn t buy stocks or Treasurys at current Jamie Dimon says mar...


In [14]:
# lowercasing
news["lower_text"] = news["normalized_text"].str.lower()

news[["normalized_text", "lower_text"]].head(3)

,normalized_text,lower_text
0,Central banks are bringing gold reserves home asgeopoliticalrisks rise Central banks are bringing gold reserves home...,central banks are bringing gold reserves home asgeopoliticalrisks rise central banks are bringing gold reserves home...
1,Oil rises 2 percent as White House says no US Iran talks happening Oil rises 2 percent as White House says no US Ira...,oil rises 2 percent as white house says no us iran talks happening oil rises 2 percent as white house says no us ira...
2,Jamie Dimon says markets underestimate risks and he wouldn t buy stocks or Treasurys at current Jamie Dimon says mar...,jamie dimon says markets underestimate risks and he wouldn t buy stocks or treasurys at current jamie dimon says mar...


In [15]:
#memastikan bahasanya english semua
def detect_language(text: str) -> str:
    if not isinstance(text, str) or len(text.strip()) < 20:
        return "unknown"
    try:
        return detect(text)
    except LangDetectException:
        return "unknown"


news["detected_lang"] = news["lower_text"].apply(detect_language)

print("Distribusi bahasa terdeteksi:")
print(news["detected_lang"].value_counts().head(10))

baris_sebelum = len(news)
news = news.loc[news["detected_lang"] == "en"].reset_index(drop=True)
print(f"\nSetelah filter bahasa Inggris (en) : {len(news)} (dibuang {baris_sebelum - len(news)} baris)")

news[["lower_text", "detected_lang"]].head(3)

Distribusi bahasa terdeteksi:
detected_lang
en    4336
Name: count, dtype: int64

Setelah filter bahasa Inggris (en) : 4336 (dibuang 0 baris)


,lower_text,detected_lang
0,central banks are bringing gold reserves home asgeopoliticalrisks rise central banks are bringing gold reserves home...,en
1,oil rises 2 percent as white house says no us iran talks happening oil rises 2 percent as white house says no us ira...,en
2,jamie dimon says markets underestimate risks and he wouldn t buy stocks or treasurys at current jamie dimon says mar...,en


## 3. Cleaning Data Kurs

In [16]:
# Gabungkan tanggal dan jam untuk satu kolom datetime kanonik.
bi_raw["Tanggal"] = pd.to_datetime(
    bi_raw["tanggal"].astype(str).str.strip() + " " + bi_raw["jam"].astype(str).str.strip(),
    format="%m/%d/%Y %I:%M %p",
    errors="coerce",
)

print(bi_raw.shape)
bi_raw.head(3)

(1202, 8)


,NO,Nilai,Kurs Jual,Kurs Beli,Tanggal_dt,tanggal,jam,Tanggal
0,1,1,17834.73,17657.27,2026-09-01,09/01/2026,12:00 AM,2026-09-01
1,2,1,17791.51,17614.49,2026-08-31,08/31/2026,12:00 AM,2026-08-31
2,3,1,17850.81,17673.19,2026-08-28,08/28/2026,12:00 AM,2026-08-28


In [17]:
def clean_numeric(series: pd.Series) -> pd.Series:
    """Hapus pemisah ribuan dan konversi nilai ke float."""
    cleaned = series.astype(str).str.replace(",", "", regex=False).str.strip()
    return pd.to_numeric(cleaned, errors="coerce")



bi_raw["Kurs Jual"] = clean_numeric(bi_raw["Kurs Jual"])
bi_raw["Kurs Beli"] = clean_numeric(bi_raw["Kurs Beli"])
bi_raw["NO"] = pd.to_numeric(bi_raw["NO"], errors="coerce")
bi_raw["Nilai"] = pd.to_numeric(bi_raw["Nilai"], errors="coerce")

print(f"Baris awal BI                 : {len(bi_raw)}")

# Pertahankan hanya rentang 1 September 2021 sampai 1 September 2026.
bi = bi_raw.loc[bi_raw["Tanggal"].between(start_date, end_date, inclusive="both")]
print(f"Setelah filter rentang tanggal: {len(bi)}")

mask_valid_numeric = (
    bi["Kurs Jual"].notna()
    & bi["Kurs Beli"].notna()
    & (bi["Kurs Jual"] > 0)
    & (bi["Kurs Beli"] > 0)
)
bi = bi.loc[mask_valid_numeric]
print(f"Setelah filter numerik valid  : {len(bi)}")

mask_valid_spread = bi["Kurs Jual"] > bi["Kurs Beli"]
bi = bi.loc[mask_valid_spread]
print(f"Setelah filter Jual > Beli    : {len(bi)}")

bi = (
    bi.sort_values("Tanggal")
    .drop_duplicates(subset=["Tanggal"], keep="last")
    .reset_index(drop=True)
)
print(f"Setelah dedup tanggal         : {len(bi)}")

bi.head(3)

Baris awal BI                 : 1202
Setelah filter rentang tanggal: 1202
Setelah filter numerik valid  : 1202
Setelah filter Jual > Beli    : 1202
Setelah dedup tanggal         : 1202


,NO,Nilai,Kurs Jual,Kurs Beli,Tanggal_dt,tanggal,jam,Tanggal
0,1202,1,14377.53,14234.47,2021-09-01,09/01/2021,12:00 AM,2021-09-01
1,1201,1,14355.42,14212.58,2021-09-02,09/02/2021,12:00 AM,2021-09-02
2,1200,1,14352.41,14209.60,2021-09-03,09/03/2021,12:00 AM,2021-09-03


# Preprocessing

In [18]:
#stopword removal
CUSTOM_STOPWORDS = {
    "cnbc", "article", "read", "also", "said", "says", "say",
    "would", "could", "also", "one", "two", "get", "got",
}
STOPWORDS_EN = set(stopwords.words("english")) | CUSTOM_STOPWORDS


def remove_stopwords(text: str) -> list:
    # Tokenisasi split-spasi lalu buang stopword Bahasa Inggris & token 1 karakter.
    if not isinstance(text, str) or not text.strip():
        return []
    tokens = text.split()
    return [tok for tok in tokens if tok not in STOPWORDS_EN and len(tok) > 1]


news["tokens_no_stopwords"] = news["lower_text"].apply(remove_stopwords)
news["text_no_stopwords"] = news["tokens_no_stopwords"].apply(" ".join)

avg_before = news["lower_text"].apply(word_count).mean()
avg_after = news["tokens_no_stopwords"].apply(len).mean()
print(f"Rata-rata jumlah kata sebelum stopword removal : {avg_before:.1f}")
print(f"Rata-rata jumlah kata sesudah stopword removal  : {avg_after:.1f}")

news[["lower_text", "text_no_stopwords"]].head(3)

Rata-rata jumlah kata sebelum stopword removal : 862.4
Rata-rata jumlah kata sesudah stopword removal  : 494.5


,lower_text,text_no_stopwords
0,central banks are bringing gold reserves home asgeopoliticalrisks rise central banks are bringing gold reserves home...,central banks bringing gold reserves home asgeopoliticalrisks rise central banks bringing gold reserves home asgeopo...
1,oil rises 2 percent as white house says no us iran talks happening oil rises 2 percent as white house says no us ira...,oil rises percent white house us iran talks happening oil rises percent white house us iran talks happening oil pric...
2,jamie dimon says markets underestimate risks and he wouldn t buy stocks or treasurys at current jamie dimon says mar...,jamie dimon markets underestimate risks buy stocks treasurys current jamie dimon markets underestimate risks buy sto...


In [19]:
#lemmatization (POS aware)
lemmatizer = WordNetLemmatizer()

POS_MAP = {
    "J": wordnet.ADJ,
    "V": wordnet.VERB,
    "N": wordnet.NOUN,
    "R": wordnet.ADV,
}


def get_wordnet_pos(treebank_tag: str) -> str:
    # Tag POS Penn Treebank (huruf pertama) ke format POS WordNet; default kata benda.
    return POS_MAP.get(treebank_tag[0], wordnet.NOUN)


def lemmatize_tokens(tokens: list) -> list:
    # Lemmatisasi tiap token berdasarkan POS tag-nya (POS-aware lemmatization).
    if not tokens:
        return []
    tagged_tokens = pos_tag(tokens)
    return [
        lemmatizer.lemmatize(token, pos=get_wordnet_pos(tag))
        for token, tag in tagged_tokens
    ]


news["tokens_lemmatized"] = news["tokens_no_stopwords"].apply(lemmatize_tokens)
news["text_lemmatized"] = news["tokens_lemmatized"].apply(" ".join)

news[["text_no_stopwords", "text_lemmatized"]].head(3)


,text_no_stopwords,text_lemmatized
0,central banks bringing gold reserves home asgeopoliticalrisks rise central banks bringing gold reserves home asgeopo...,central bank bring gold reserve home asgeopoliticalrisks rise central bank bring gold reserve home asgeopoliticalris...
1,oil rises percent white house us iran talks happening oil rises percent white house us iran talks happening oil pric...,oil rise percent white house u iran talk happen oil rise percent white house u iran talk happen oil price rise thurs...
2,jamie dimon markets underestimate risks buy stocks treasurys current jamie dimon markets underestimate risks buy sto...,jamie dimon market underestimate risk buy stock treasurys current jamie dimon market underestimate risk buy stock tr...


# Data Labelling
## VADER

In [20]:
# ===== VADER: skor sentimen JUDUL =====
USE_VADER = True      # True = aktif, False = nonaktif
THR = 0.05            # ambang standar VADER untuk compound

from nltk.sentiment.vader import SentimentIntensityAnalyzer

nltk.download('vader_lexicon', quiet=True)
vader = SentimentIntensityAnalyzer()

VADER_COLS = ['vader_title', 'vader_title_pos', 'vader_title_neg', 'vader_title_neu']

def score_title(t):
    # judul kosong -> semua NaN (bukan 0 / netral)
    if isinstance(t, str) and t.strip():
        s = vader.polarity_scores(t)                      # dipanggil sekali per judul
        return [s['compound'], s['pos'], s['neg'], s['neu']]
    return [np.nan] * 4

# bersihkan sisa run sebelumnya
news = news.drop(columns=[c for c in news.columns if c.startswith('vader_title')])

if USE_VADER:
    # pakai title ASLI (bukan lower_text / lemma): VADER membaca huruf kapital, tanda seru, dan negasi
    v = pd.DataFrame(news['title'].apply(score_title).tolist(), columns=VADER_COLS, index=news.index)
    news = pd.concat([news, v], axis=1)

    # polaritas dari compound: > 0.05 -> +1, < -0.05 -> -1, sisanya 0. NaN tetap NaN.
    comp = news['vader_title']
    pol = np.select([comp > THR, comp < -THR], [1, -1], default=0).astype(float)
    news['vader_title_polarity'] = pd.Series(pol, index=news.index).where(comp.notna())

    valid = news['vader_title'].dropna()
    print('VADER aktif')
    print(f"Total: {len(news)} | NaN: {news['vader_title'].isna().sum()} -> tetap NaN")
    print("Distribusi polaritas (dari judul valid):")
    print(news['vader_title_polarity'].value_counts(normalize=True).sort_index().round(3).to_string())
    print(f"Rata-rata proporsi kata | pos: {news['vader_title_pos'].mean():.3f} | "
          f"neg: {news['vader_title_neg'].mean():.3f} | neu: {news['vader_title_neu'].mean():.3f}")
    display(news[['title'] + VADER_COLS + ['vader_title_polarity']].head())
else:
    print('VADER dinonaktifkan')

VADER aktif
Total: 4336 | NaN: 0 -> tetap NaN
Distribusi polaritas (dari judul valid):
vader_title_polarity
-1.0    0.444
 0.0    0.293
 1.0    0.263
Rata-rata proporsi kata | pos: 0.076 | neg: 0.123 | neu: 0.800


,title,vader_title,vader_title_pos,vader_title_neg,vader_title_neu,vader_title_polarity
0,Central banks are bringing gold reserves home asgeopoliticalrisks rise,0.0000,0.000,0.000,1.000,0.0
1,Oil rises 2% as White House says no US-Iran talks happening,-0.2960,0.000,0.180,0.820,-1.0
2,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ...,-0.5106,0.000,0.235,0.765,-1.0
3,China’s super-rich fled Singapore. Now they want to come back,0.0772,0.126,0.000,0.874,1.0
4,‘Don’t get too comfortable’: Wall Street’s ‘fear gauge’ hits 2026 low — here’s why it’s ...,-0.2732,0.000,0.130,0.870,-1.0


In [21]:
#tampilkan published_date yang NaN
news[news['published_date'].isna()]

#drop baris yang published_date nya NaN
news = news.dropna(subset=['published_date']).reset_index(drop=True)

## Loughran-McDonald

In [22]:
# ===== Loughran-McDonald pada lower_text =====
LM_PATH = 'data/raw/Loughran-McDonald_MasterDictionary_1993-2025.csv'   # sesuaikan nama file

lm = pd.read_csv(LM_PATH)
lm.columns = [c.strip() for c in lm.columns]
lm['Word'] = lm['Word'].astype(str).str.lower()

# cocokkan nama kolom tanpa peduli underscore/kapital, jadi tahan terhadap beda versi file
norm = {c.lower().replace('_', '').replace(' ', ''): c for c in lm.columns}
LM_CATS = {'neg': 'negative', 'pos': 'positive', 'unc': 'uncertainty', 'lit': 'litigious',
           'strong': 'strongmodal', 'weak': 'weakmodal', 'constr': 'constraining'}
missing = [v for v in LM_CATS.values() if v not in norm]
assert not missing, f"kolom tidak ditemukan: {missing}. Kolom tersedia: {list(lm.columns)}"

# kolom kategori bernilai 0 jika bukan anggota, selain itu berisi tahun kata ditambahkan
LM_SETS = {k: frozenset(lm.loc[lm[norm[v]] > 0, 'Word']) for k, v in LM_CATS.items()}
print({k: len(v) for k, v in LM_SETS.items()})

def lm_score(text):
    toks = text.split()
    n = len(toks)
    if n == 0:
        return {'lm_text_n_tokens': 0, **{f'lm_text_{k}': np.nan for k in LM_SETS},
                'lm_text_net': np.nan, 'lm_text_polarity': np.nan}
    cnt = {k: sum(t in s for t in toks) for k, s in LM_SETS.items()}
    p, ng = cnt['pos'], cnt['neg']
    out = {'lm_text_n_tokens': n}
    out.update({f'lm_text_{k}': v / n for k, v in cnt.items()})            # proporsi terhadap total token
    out['lm_text_net'] = (p - ng) / n                                       # (pos - neg) / total token
    out['lm_text_polarity'] = (p - ng) / (p + ng) if (p + ng) > 0 else 0.0  # skala -1..1
    return out

scores = pd.DataFrame([lm_score(t) for t in news['lower_text']], index=news.index)
news = news.drop(columns=[c for c in scores.columns if c in news.columns]).join(scores)   # aman dijalankan ulang

# ---------- Cek ----------
LM_COLS = list(scores.columns)
print(f"Artikel: {len(news)} | lower_text kosong: {(news['lm_text_n_tokens'] == 0).sum()}")
display(news[LM_COLS].describe().T[['mean', 'std', 'min', '50%', 'max']])

if 'vader_title' in news.columns:
    print("Korelasi lm_text_net vs vader_title:", round(news['lm_text_net'].corr(news['vader_title']), 3))

# kata LM paling sering muncul (sanity check: harus masuk akal untuk berita geopolitik-ekonomi)
tok_counter = Counter(t for txt in news['lower_text'] for t in txt.split())
for k in ('neg', 'pos', 'unc'):
    top = [(w, c) for w, c in tok_counter.most_common() if w in LM_SETS[k]][:12]
    print(f"\nTop {k}:", top)

print("\nArtikel paling negatif:")
display(news.nsmallest(5, 'lm_text_net')[['title', 'lm_text_net']])
print("Artikel paling positif:")
display(news.nlargest(5, 'lm_text_net')[['title', 'lm_text_net']])

{'neg': 2345, 'pos': 347, 'unc': 297, 'lit': 903, 'strong': 19, 'weak': 27, 'constr': 184}
Artikel: 4335 | lower_text kosong: 0


,mean,std,min,50%,max
lm_text_n_tokens,862.495732,426.637639,396.000000,775.000000,12435.000000
lm_text_neg,0.022857,0.010865,0.000000,0.021711,0.074394
lm_text_pos,0.007417,0.005132,0.000000,0.006467,0.037037
lm_text_unc,0.009524,0.006011,0.000000,0.008446,0.042654
lm_text_lit,0.002581,0.005153,0.000000,0.001213,0.086498
lm_text_strong,0.004405,0.003524,0.000000,0.003683,0.025581
lm_text_weak,0.004765,0.003598,0.000000,0.004152,0.035545
lm_text_constr,0.001912,0.002280,0.000000,0.001384,0.024869
lm_text_net,-0.015440,0.013053,-0.072664,-0.014717,0.036000
lm_text_polarity,-0.475604,0.351321,-1.000000,-0.534884,1.000000


Korelasi lm_text_net vs vader_title: 0.37

Top neg: [('conflict', 4088), ('against', 3876), ('concerns', 1851), ('cut', 1487), ('crisis', 1366), ('warned', 1153), ('threat', 919), ('fears', 842), ('volatility', 718), ('losses', 663), ('disruption', 655), ('recession', 633)]

Top pos: [('despite', 1514), ('strong', 1298), ('good', 1115), ('better', 904), ('gains', 903), ('able', 771), ('highest', 752), ('best', 733), ('leading', 663), ('boost', 636), ('leadership', 611), ('alliance', 610)]

Top unc: [('could', 7151), ('may', 3738), ('risk', 2959), ('risks', 1681), ('nearly', 1353), ('uncertainty', 1067), ('might', 981), ('possible', 953), ('almost', 819), ('volatility', 718), ('believe', 696), ('roughly', 628)]

Artikel paling negatif:


,title,lm_text_net
2719,Defense Department warns climate change will increase conflicts over water and food,-0.072664
2610,Oil prices climb more than $1 on fear of spreading Middle Eastconflict,-0.068337
1230,"‘Aggressive’ China cyberattacks are the ‘defining threat’ of our time, top U.S. cyber official says",-0.067416
2385,"Binance CEO Changpeng Zhao pleads guilty to federal charges, steps down",-0.066510
2205,Harvard professors warn that war-torn countries will miss global vaccine goals in 2022,-0.061798


Artikel paling positif:


,title,lm_text_net
1405,"Here are the top 10 U.S. stock analysts, according to TipRanks",0.036000
1314,"Geopoliticaltensionswith the U.S. could ‘supercharge’ China’s innovation, JPMorgan says",0.030501
1503,Top Wall Street analysts favor these stocks for attractive long-term potential,0.028302
3225,Top Wall Street analysts prefer these dividend stocks for boosting portfolio returns,0.026911
1519,Top Wall Street analysts are confident about the potential behind these 3 stocks,0.025826


# Penyelarasan Data & Agregasi Harian

Alur bagian ini:
1. **Alignment**: artikel yang terbit setelah cutoff **15:00 WIB** digeser ke hari bursa berikutnya; weekend dan libur nasional otomatis ikut ke hari bursa BI berikutnya.
2. **Agregasi per hari bursa**: skor VADER dan Loughran–McDonald dipool dari artikel harian, sementara volume berita disimpan sebagai fitur tersendiri.
3. **Fitur teks harian** disiapkan untuk eksperimen TF-IDF.

Tidak digunakan **time-weighting / feature weighting manual**.


In [23]:
# ===== Alignment ke hari bursa BI =====
CUTOFF = time(15, 0)     # setelah 15:00 WIB -> hari bursa berikutnya (tepat 15:00:00 masih hari yang sama)

# timestamp kanonik: published_raw -> WIB (naif). Dipakai untuk alignment, bobot waktu, dan cek lain.
news['ts_wib'] = (pd.to_datetime(news['published_raw'], errors='coerce', utc=True)
                    .dt.tz_convert('Asia/Jakarta').dt.tz_localize(None))
n_nat = int(news['ts_wib'].isna().sum())
news = news.dropna(subset=['ts_wib']).reset_index(drop=True)

after = news['ts_wib'].dt.time.map(lambda t: t > CUTOFF)
news['cand_date'] = (news['ts_wib'].dt.normalize() + pd.to_timedelta(after.astype(int), unit='D')).astype('datetime64[ns]')
news['shifted'] = after.astype(int)

# lompat ke tanggal BI terdekat setelahnya -> weekend & libur nasional otomatis tergabung ke hari bursa berikutnya
bi_dates = (bi[['Tanggal_dt']].rename(columns={'Tanggal_dt': 'aligned_trading_date'})
              .astype({'aligned_trading_date': 'datetime64[ns]'}).sort_values('aligned_trading_date'))

news = news.drop(columns=['aligned_trading_date'], errors='ignore')
news = pd.merge_asof(news.sort_values('cand_date'), bi_dates,
                     left_on='cand_date', right_on='aligned_trading_date', direction='forward')

n_out = int(news['aligned_trading_date'].isna().sum())
news = news.dropna(subset=['aligned_trading_date']).sort_values('ts_wib').reset_index(drop=True)

# ---------- Cek ----------
print(f"ts_wib NaT (dibuang): {n_nat} | di luar rentang BI (dibuang): {n_out} | artikel tersisa: {len(news)}")
print(f"Digeser karena lewat {CUTOFF}: {news['shifted'].mean():.1%}")
print("Hari aligned (0=Senin):", news['aligned_trading_date'].dt.weekday.value_counts().sort_index().to_dict())
print("Loncat kalender (hari) cand_date -> aligned:",
      (news['aligned_trading_date'] - news['cand_date']).dt.days.value_counts().sort_index().to_dict())
if 'date_raw_dt_wib' in news.columns:
    gap_h = (news['ts_wib'] - news['date_raw_dt_wib']).dt.total_seconds() / 3600
    print("ts_wib (published_raw) - date_raw_dt_wib, dalam jam:"); print(gap_h.describe().round(2).to_string())

ts_wib NaT (dibuang): 0 | di luar rentang BI (dibuang): 3 | artikel tersisa: 4332
Digeser karena lewat 15:00:00: 47.4%
Hari aligned (0=Senin): {0: 1234, 1: 759, 2: 880, 3: 733, 4: 726}
Loncat kalender (hari) cand_date -> aligned: {0: 3313, 1: 344, 2: 455, 3: 88, 4: 49, 5: 21, 6: 27, 7: 22, 8: 1, 9: 2, 10: 7, 11: 3}
ts_wib (published_raw) - date_raw_dt_wib, dalam jam:
count    4332.00
mean       -6.99
std         0.49
min       -25.55
25%        -7.00
50%        -7.00
75%        -7.00
max         8.51


In [24]:
# ===== Agregasi per aligned_trading_date =====
# hitungan kata LM per artikel
tok = news['lm_text_n_tokens']
news['lm_pos_cnt'] = (news['lm_text_pos'] * tok).round()
news['lm_neg_cnt'] = (news['lm_text_neg'] * tok).round()
news['lm_unc_cnt'] = (news['lm_text_unc'] * tok).round()

g = news.groupby('aligned_trading_date')
daily = g.agg(
    news_count   =('title', 'size'),
    share_shifted=('shifted', 'mean'),
    # VADER (judul)
    vader_mean   =('vader_title', 'mean'),
    vader_min    =('vader_title', 'min'),
    vader_max    =('vader_title', 'max'),
    vader_std    =('vader_title', 'std'),
    vader_pol_sum=('vader_title_polarity', 'sum'),
    vader_pos_n  =('vader_title_polarity', lambda x: (x > 0).sum()),
    vader_neg_n  =('vader_title_polarity', lambda x: (x < 0).sum()),
    # LM (lower_text): polaritas per artikel + jumlah untuk pooling
    si_lm_art    =('lm_text_polarity', 'mean'),
    lm_pos_sum   =('lm_pos_cnt', 'sum'),
    lm_neg_sum   =('lm_neg_cnt', 'sum'),
    lm_unc_sum   =('lm_unc_cnt', 'sum'),
    lm_tok_sum   =('lm_text_n_tokens', 'sum'),
).reset_index()

N = daily['news_count']
daily['vader_pos_ratio'] = daily['vader_pos_n'] / N
daily['vader_neg_ratio'] = daily['vader_neg_n'] / N
daily['si_vader']        = daily['vader_pol_sum'] / N                                   # (n_pos - n_neg) / N
daily['lm_neg_w']        = daily['lm_neg_sum'] / daily['lm_tok_sum']                    # rasio dipool per token
daily['lm_pos_w']        = daily['lm_pos_sum'] / daily['lm_tok_sum']
daily['lm_unc_w']        = daily['lm_unc_sum'] / daily['lm_tok_sum']
daily['lm_net_w']        = (daily['lm_pos_sum'] - daily['lm_neg_sum']) / daily['lm_tok_sum']
daily['lm_polarity_w']   = ((daily['lm_pos_sum'] - daily['lm_neg_sum'])
                            / (daily['lm_pos_sum'] + daily['lm_neg_sum']).replace(0, np.nan))
# ---------- Merge ke kalender BI (satu baris per hari bursa) ----------
df = bi.merge(daily, how='left', left_on='Tanggal_dt', right_on='aligned_trading_date',
              validate='one_to_one').drop(columns='aligned_trading_date')
df = df.sort_values('Tanggal_dt').reset_index(drop=True)

df['news_count'] = df['news_count'].fillna(0).astype(int)
df['has_news']   = (df['news_count'] > 0).astype(int)
df['gap_days']   = df['Tanggal_dt'].diff().dt.days.fillna(1).astype(int)   # selisih kalender dari hari bursa sebelumnya
df['news_per_cal_day'] = df['news_count'] / df['gap_days']

FILL0 = [c for c in df.columns if c.startswith(('vader_', 'lm_', 'si_')) or c == 'share_shifted']
df[FILL0] = df[FILL0].fillna(0.0)           # hari tanpa berita: netral, dibedakan oleh has_news / news_count

# ---------- Rolling (kausal) dari jumlah, bukan rata-rata rata-rata ----------
def roll_ratio(num, den, w):
    n_, d_ = df[num].rolling(w, min_periods=1).sum(), df[den].rolling(w, min_periods=1).sum()
    return (n_ / d_.replace(0, np.nan)).fillna(0.0)

for w in (3, 5):
    df[f'si_vader_r{w}']   = roll_ratio('vader_pol_sum', 'news_count', w)
    df[f'lm_net_w_r{w}']   = roll_ratio('lm_pos_sum', 'lm_tok_sum', w) - roll_ratio('lm_neg_sum', 'lm_tok_sum', w)
    df[f'news_count_r{w}'] = df['news_count'].rolling(w, min_periods=1).sum()

# ---------- Cek ----------
assert df['news_count'].sum() == len(news), "ada artikel yang hilang saat merge ke BI"
print(f"Hari bursa: {len(df)} | tanpa berita: {(df['has_news'] == 0).sum()} ({(df['has_news'] == 0).mean():.1%})")
print("Rata-rata news_count per gap_days:"); print(df.groupby('gap_days')['news_count'].agg(['mean', 'count']).round(2).to_string())
print("\n5 hari dengan berita terbanyak (harus Senin/pasca-libur, bukan bug):")
display(df.nlargest(5, 'news_count')[['Tanggal_dt', 'gap_days', 'news_count', 'si_vader', 'lm_net_w']])
print("Korelasi si_vader vs lm_net_w:", round(df.loc[df['has_news'] == 1, ['si_vader', 'lm_net_w']].corr().iloc[0, 1], 3))

Hari bursa: 1202 | tanpa berita: 135 (11.2%)
Rata-rata news_count per gap_days:
           mean  count
gap_days              
1          2.89    923
2          5.22     18
3          4.83    223
4         10.33     18
5          9.92     13
6          3.00      2
8         47.50      2
11        27.00      2
12        26.00      1

5 hari dengan berita terbanyak (harus Senin/pasca-libur, bukan bug):


,Tanggal_dt,gap_days,news_count,si_vader,lm_net_w
1097,2026-03-25,8,80,-0.425000,-0.017411
635,2024-04-16,11,39,-0.282051,-0.019917
914,2025-06-23,3,29,-0.586207,-0.020587
126,2022-03-01,4,28,0.071429,-0.019323
868,2025-04-08,12,26,-0.230769,-0.018091


Korelasi si_vader vs lm_net_w: 0.31


In [25]:
# ===== Teks harian (TF-IDF dari teks asli, tanpa bobot waktu) =====
extra = (
    news.groupby('aligned_trading_date')['lower_text']
         .agg(' '.join)
         .rename('all_lower_text')
         .reset_index()
         .rename(columns={'aligned_trading_date': 'Tanggal_dt'})
)

df = df.drop(columns=['all_lower_text'], errors='ignore')
df = df.merge(
    extra,
    on='Tanggal_dt',
    how='left',
    validate='one_to_one'
)
df['all_lower_text'] = df['all_lower_text'].fillna('')

print("Teks harian berhasil dibuat tanpa time-weighting.")


Teks harian berhasil dibuat tanpa time-weighting.


# Modelling: Prediksi Return dengan TF-IDF + SVD + Perbandingan Metode Pelabelan

**Tujuan eksperimen:** menentukan apakah dan seberapa besar metode pelabelan berita memengaruhi prediksi `return` kurs USD/IDR.

- **Target:** `y = return(t+1)` kontinu, bukan kelas 0/1.
- **Embedding teks:** **TF-IDF** dengan konfigurasi yang sama pada seluruh eksperimen, lalu direduksi dengan **TruncatedSVD (LSA)**.
- **Yang dibandingkan:** metode pelabelan/sinyal berita, bukan jenis embedding.
- **Model ML:** **satu model regresi (Ridge) dengan parameter default**, tanpa tuning hyperparameter.
- **Split:** kronologis 70/15/15 tanpa shuffle, dibuat **sebelum** TF-IDF dan SVD (keduanya di-fit hanya pada train).
- **Metrik:** RMSE, MAE, R², R² OOS vs zero, Directional Accuracy, dan **F1** (macro-F1 arah naik/turun dari prediksi return).
- **Test:** hanya untuk evaluasi akhir setelah refit train+validation.

Dengan desain ini, baseline `No label` (TF-IDF+SVD dan fitur time-series/volume) menjadi pembanding utama. Tambahan VADER dan/atau LM menunjukkan **incremental effect** dari metode pelabelan ketika representasi teks sudah tetap.

In [26]:
# ===== TARGET DAN FITUR UMUM =====
H = 1

# Kurs tengah BI
df['mid'] = (df['Kurs Jual'] + df['Kurs Beli']) / 2

# Return harian dalam persen
df['ret'] = 100 * np.log(df['mid']).diff()

# Target kontinu: return hari berikutnya
df['y'] = df['ret'].shift(-H)

# ---------------------------------------------------------
# Fitur umum yang SAMA untuk semua eksperimen
# ---------------------------------------------------------
for k in range(5):
    df[f'ret_lag{k}'] = df['ret'].shift(k)

df['vol_5'] = df['ret'].rolling(5).std()
df['vol_20'] = df['ret'].rolling(20).std()
df['mom_5'] = df['ret'].rolling(5).sum()
df['ret_roll_mean5'] = df['ret'].rolling(5).mean()
df['ret_roll_std5'] = df['ret'].rolling(5).std()

COMMON_COLS = (
    [f'ret_lag{k}' for k in range(5)]
    + ['vol_5', 'vol_20', 'mom_5', 'ret_roll_mean5', 'ret_roll_std5',
       'news_count', 'has_news', 'gap_days', 'news_per_cal_day',
       'news_count_r3', 'news_count_r5', 'share_shifted']
)

# ---------------------------------------------------------
# Satu set fitur untuk masing-masing metode pelabelan
# Hanya sinyal inti + lag/rolling/diff, agar efek metode label
# dapat dibandingkan dengan struktur fitur yang konsisten.
# ---------------------------------------------------------
LABEL_FEATURES = {
    'No label': [],
    'VADER': [
        'si_vader', 'si_vader_lag1', 'si_vader_roll3', 'si_vader_diff1'
    ],
    'Loughran-McDonald': [
        'lm_net_w', 'lm_net_w_lag1', 'lm_net_w_roll3', 'lm_net_w_diff1'
    ],
    'VADER + LM': [
        'si_vader', 'si_vader_lag1', 'si_vader_roll3', 'si_vader_diff1',
        'lm_net_w', 'lm_net_w_lag1', 'lm_net_w_roll3', 'lm_net_w_diff1'
    ],
}
# Filter dilakukan setelah fitur lag/rolling/diff dibuat pada cell berikutnya.


In [27]:
# ===== BUAT LAG/ROLLING/DIFF PADA SINYAL LABEL, LALU SAMPEL FINAL (ds) =====
# Fitur inti diambil dari hari t lalu diturunkan hanya dari masa lalu.
CORE_LABEL_SIGNALS = [
    c for c in ['si_vader', 'lm_net_w']
    if c in df.columns
]

for c in CORE_LABEL_SIGNALS:
    df[f'{c}_lag1'] = df[c].shift(1)
    df[f'{c}_roll3'] = df[c].rolling(3, min_periods=1).mean()
    df[f'{c}_diff1'] = df[c].diff()

# Sekarang filter fitur label setelah seluruh fitur turunan tersedia.
LABEL_FEATURES = {
    name: [c for c in cols if c in df.columns]
    for name, cols in LABEL_FEATURES.items()
}

# Sampel final: satu kali saja, setelah SEMUA fitur turunan tersedia,
# supaya seluruh eksperimen memakai sampel kronologis yang sama.
ALL_EXPERIMENT_COLS = list(dict.fromkeys(
    COMMON_COLS + [c for cols in LABEL_FEATURES.values() for c in cols]
))
ALL_EXPERIMENT_COLS = [c for c in ALL_EXPERIMENT_COLS if c in df.columns]
ds = df.dropna(subset=['y'] + ALL_EXPERIMENT_COLS).reset_index(drop=True)

print('Core label signals:', CORE_LABEL_SIGNALS)
print('Jumlah observasi (ds):', len(ds))
print('Fitur umum:', len([c for c in COMMON_COLS if c in ds.columns]))
for name, cols in LABEL_FEATURES.items():
    print(f'{name:20}: {len(cols)} fitur label')


Core label signals: ['si_vader', 'lm_net_w']
Jumlah observasi (ds): 1181
Fitur umum: 17
No label            : 0 fitur label
VADER               : 4 fitur label
Loughran-McDonald   : 4 fitur label
VADER + LM          : 8 fitur label


## Split kronologis 70/15/15

Dilakukan **sebelum** TF-IDF: vocabulary dan bobot IDF hanya boleh dipelajari dari data train.

In [28]:
# ===== SPLIT KRONOLOGIS 70/15/15 (SEBELUM TF-IDF) =====
# ds terurut menaik menurut Tanggal_dt, jadi potongan berurutan = split kronologis tanpa shuffle.
assert ds['Tanggal_dt'].is_monotonic_increasing, 'ds harus terurut menaik menurut tanggal'

n = len(ds)
i1, i2 = int(n * 0.70), int(n * 0.85)
tr   = np.arange(0, i1)
va   = np.arange(i1, i2)
te   = np.arange(i2, n)
trva = np.concatenate([tr, va])          # dipakai untuk refit final sebelum evaluasi test

train_df, val_df, test_df = ds.iloc[tr], ds.iloc[va], ds.iloc[te]

# cek: urutan waktu benar dan tidak ada tumpang tindih
assert train_df['Tanggal_dt'].max() < val_df['Tanggal_dt'].min()
assert val_df['Tanggal_dt'].max() < test_df['Tanggal_dt'].min()
assert len(tr) + len(va) + len(te) == n

print('Jumlah observasi:', n)
for name, d in [('Train', train_df), ('Validation', val_df), ('Test', test_df)]:
    print(
        f'{name:10}: {len(d):4d} ({len(d) / n:.0%}) | '
        f'{d["Tanggal_dt"].min().date()} -> {d["Tanggal_dt"].max().date()} | '
        f'mean y={d["y"].mean():.6f} | hari naik (y>0)={(d["y"] > 0).mean():.1%}'
    )

Jumlah observasi: 1181
Train     :  826 (70%) | 2021-09-29 -> 2025-02-25 | mean y=0.016230 | hari naik (y>0)=55.8%
Validation:  177 (15%) | 2025-02-26 -> 2025-11-25 | mean y=0.012025 | hari naik (y>0)=52.0%
Test      :  178 (15%) | 2025-11-26 -> 2026-08-31 | mean y=0.035241 | hari naik (y>0)=58.4%


In [29]:
# ===== TF-IDF: SATU EMBEDDING UNTUK SEMUA EKSPERIMEN =====
# (split 70/15/15 -> tr, va, te, trva sudah dibuat di cell sebelumnya)
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
from scipy.sparse import hstack, csr_matrix
from sklearn.preprocessing import StandardScaler

STOP = sorted(
    set(stopwords.words('english')) |
    set(globals().get('CUSTOM_STOPWORDS', set()))
)

TFIDF_PARAMS = dict(
    stop_words=STOP,
    max_features=2000,
    ngram_range=(1, 2),
    min_df=5,
    sublinear_tf=True
)

# TF-IDF di-fit hanya pada TRAIN. Vocabulary dan bobotnya lalu dipakai
# untuk validation/test sehingga tidak terjadi data leakage.
tfidf_tr = TfidfVectorizer(**TFIDF_PARAMS)
Tf_tr = tfidf_tr.fit_transform(ds.loc[tr, 'all_lower_text'].fillna(''))
Tf_va = tfidf_tr.transform(ds.loc[va, 'all_lower_text'].fillna(''))

# Untuk evaluasi test final, fit ulang TF-IDF pada train+validation.
tfidf_trva = TfidfVectorizer(**TFIDF_PARAMS)
Tf_trva = tfidf_trva.fit_transform(ds.loc[trva, 'all_lower_text'].fillna(''))
Tf_te = tfidf_trva.transform(ds.loc[te, 'all_lower_text'].fillna(''))

print('TF-IDF vocabulary:', len(tfidf_tr.vocabulary_))
print('Train shape:', Tf_tr.shape)
print('Validation shape:', Tf_va.shape)
print('Train+Val shape:', Tf_trva.shape)
print('Test shape:', Tf_te.shape)


TF-IDF vocabulary: 2000
Train shape: (826, 2000)
Validation shape: (177, 2000)
Train+Val shape: (1003, 2000)
Test shape: (178, 2000)


## Reduksi dimensi: TruncatedSVD (LSA)

TF-IDF (2.000 fitur, sparse) direduksi menjadi `SVD_COMPONENTS` komponen dense. Seperti TF-IDF, SVD di-fit **hanya pada data fit**: train untuk validasi, train+validation untuk evaluasi test final.

In [30]:
# ===== REDUKSI DIMENSI: TruncatedSVD (LSA) PADA TF-IDF =====
from sklearn.decomposition import TruncatedSVD

SVD_COMPONENTS = 100
SVD_SEED = 42        # hanya untuk reproduksibilitas (solver randomized)
SCALE_SVD = True     # standarisasi komponen SVD (di-fit pada data fit) agar skalanya sebanding dengan blok numerik

def fit_svd(T_fit, T_other):
    """Fit SVD pada T_fit saja, lalu transform T_fit dan T_other."""
    k = min(SVD_COMPONENTS, T_fit.shape[0] - 1, T_fit.shape[1] - 1)
    svd = TruncatedSVD(n_components=k, random_state=SVD_SEED).fit(T_fit)
    Z_fit, Z_other = svd.transform(T_fit), svd.transform(T_other)
    if SCALE_SVD:
        sc = StandardScaler().fit(Z_fit)
        Z_fit, Z_other = sc.transform(Z_fit), sc.transform(Z_other)
    return svd, Z_fit, Z_other

# Validasi: SVD di-fit pada TRAIN saja
svd_tr, Z_tr, Z_va = fit_svd(Tf_tr, Tf_va)
# Test final: SVD di-fit ulang pada TRAIN+VALIDATION
svd_trva, Z_trva, Z_te = fit_svd(Tf_trva, Tf_te)

print('Komponen SVD:', svd_tr.n_components)
print('Varians terjelaskan (train):', f'{svd_tr.explained_variance_ratio_.sum():.1%}')
print('Varians terjelaskan (train+val):', f'{svd_trva.explained_variance_ratio_.sum():.1%}')
print('Shape Z_tr / Z_va / Z_trva / Z_te:', Z_tr.shape, Z_va.shape, Z_trva.shape, Z_te.shape)

# Istilah dengan bobot terbesar pada 5 komponen pertama (sanity check topik)
terms = np.array(tfidf_tr.get_feature_names_out())
for k in range(5):
    top = terms[np.argsort(-np.abs(svd_tr.components_[k]))[:8]]
    print(f'Komponen {k + 1}:', ', '.join(top))

Komponen SVD: 100
Varians terjelaskan (train): 48.4%
Varians terjelaskan (train+val): 45.8%
Shape Z_tr / Z_va / Z_trva / Z_te: (826, 100) (177, 100) (1003, 100) (178, 100)
Komponen 1: percent, china, russia, ukraine, year, dollar, war, global
Komponen 2: percent, russian, putin, military, prices, russia, ukraine, dollar
Komponen 3: israel, gaza, hamas, israeli, oil, crude, iran, hezbollah
Komponen 4: ukraine, russian, russia, israel, putin, gaza, hamas, israeli
Komponen 5: china, beijing, chinese, xi, taiwan, oil, crude, opec


## Model comparison

Untuk setiap metode pelabelan, matriks input selalu terdiri dari:

**komponen SVD dari TF-IDF + fitur umum time-series/news volume + fitur label yang sedang diuji.**

Jadi perbedaannya bukan pada embedding. Satu model (Ridge, parameter default) menjalani eksperimen pada semua metode label.

In [31]:
# ===== BANDINGKAN BANYAK MODEL (PARAMETER DEFAULT) DI SETIAP METODE PELABELAN =====
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression, Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, f1_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

SEED = 42   # hanya untuk reproduksibilitas, bukan hyperparameter

# Semua model memakai parameter default library: tidak ada grid dan tidak ada tuning.
MODELS = {
    'LinearRegression': lambda: LinearRegression(),
    'Ridge':            lambda: Ridge(),
    'ElasticNet':       lambda: ElasticNet(),
    'RandomForest':     lambda: RandomForestRegressor(random_state=SEED, n_jobs=-1),
    'ExtraTrees':       lambda: ExtraTreesRegressor(random_state=SEED, n_jobs=-1),
    'XGBoost':          lambda: XGBRegressor(random_state=SEED, n_jobs=-1, verbosity=0),
    'LightGBM':         lambda: LGBMRegressor(random_state=SEED, n_jobs=-1, verbosity=-1),
}

def evaluate_regression(y, p):
    y = np.asarray(y, dtype=float)
    p = np.asarray(p, dtype=float)
    # Arah untuk F1: naik (return > 0) vs turun/tetap (return <= 0)
    y_up = (y > 0).astype(int)
    p_up = (p > 0).astype(int)
    return {
        'RMSE': np.sqrt(mean_squared_error(y, p)),
        'MAE': mean_absolute_error(y, p),
        'R2': r2_score(y, p),
        'R2_oos_vs_zero': 1 - np.sum((y - p) ** 2) / np.sum(y ** 2),
        'DirAcc': np.mean(np.sign(y) == np.sign(p)),
        'F1': f1_score(y_up, p_up, average='macro', zero_division=0),   # macro-F1 arah naik/turun
        'pred_std': np.std(p),    # ~0 = model runtuh jadi prediksi konstan
    }

# Fitur numerik umum + label. Blok numerik distandardisasi sendiri (di-fit pada data fit);
# komponen SVD sudah distandardisasi di cell sebelumnya (SCALE_SVD).
COMMON_COLS = [c for c in COMMON_COLS if c in ds.columns]

y_tr = ds.loc[tr, 'y'].to_numpy()
y_va = ds.loc[va, 'y'].to_numpy()
y_trva = ds.loc[trva, 'y'].to_numpy()
y_te = ds.loc[te, 'y'].to_numpy()

results_val = []
results_test = []

for label_name, label_cols in LABEL_FEATURES.items():
    feature_cols = list(dict.fromkeys(COMMON_COLS + [c for c in label_cols if c in ds.columns]))

    Xnum_tr = ds.loc[tr, feature_cols].to_numpy(dtype=float)
    Xnum_va = ds.loc[va, feature_cols].to_numpy(dtype=float)
    Xnum_trva = ds.loc[trva, feature_cols].to_numpy(dtype=float)
    Xnum_te = ds.loc[te, feature_cols].to_numpy(dtype=float)

    scaler = StandardScaler().fit(Xnum_tr)
    scaler_trva = StandardScaler().fit(Xnum_trva)

    # [fitur numerik | komponen SVD]  (dense); dibuat sekali per label, dipakai semua model
    X_tr = np.hstack([scaler.transform(Xnum_tr), Z_tr])
    X_va = np.hstack([scaler.transform(Xnum_va), Z_va])
    X_trva = np.hstack([scaler_trva.transform(Xnum_trva), Z_trva])
    X_te = np.hstack([scaler_trva.transform(Xnum_te), Z_te])

    for model_name, make_model in MODELS.items():
        # Validasi: fit pada train
        model = make_model().fit(X_tr, y_tr)
        results_val.append({
            'labeling': label_name, 'model': model_name,
            'n_label_features': len(label_cols), 'best_params': 'default',
            **evaluate_regression(y_va, model.predict(X_va)),
        })

        # Test final: refit pada train+validation
        final_model = make_model().fit(X_trva, y_trva)
        results_test.append({
            'labeling': label_name, 'model': model_name,
            'n_label_features': len(label_cols), 'best_params': 'default',
            **evaluate_regression(y_te, final_model.predict(X_te)),
        })

val_results = pd.DataFrame(results_val)
test_results = pd.DataFrame(results_test)

pd.set_option('display.max_colwidth', 60)
pd.set_option('display.float_format', '{:.6f}'.format)

print(f'{len(MODELS)} model x {len(LABEL_FEATURES)} metode label | parameter default | '
      f'fitur terakhir: {X_tr.shape[1]} (numerik + {Z_tr.shape[1]} komponen SVD)')
print('VALIDATION — fit pada train')
display(val_results.sort_values(['labeling', 'RMSE']).reset_index(drop=True))

print('TEST — refit train+validation')
display(test_results.sort_values(['labeling', 'RMSE']).reset_index(drop=True))

7 model x 4 metode label | parameter default | fitur terakhir: 125 (numerik + 100 komponen SVD)
VALIDATION — fit pada train


,labeling,model,n_label_features,best_params,RMSE,MAE,R2,R2_oos_vs_zero,DirAcc,F1,pred_std
0,Loughran-McDonald,ExtraTrees,4,default,0.323972,0.244891,0.086284,0.087433,0.491525,0.492705,0.069501
1,Loughran-McDonald,RandomForest,4,default,0.331497,0.248626,0.043346,0.044549,0.531073,0.512231,0.053212
2,Loughran-McDonald,LinearRegression,4,default,0.332015,0.255542,0.040352,0.041559,0.525424,0.535522,0.116969
3,Loughran-McDonald,Ridge,4,default,0.332063,0.255550,0.040079,0.041285,0.525424,0.535522,0.116515
4,Loughran-McDonald,ElasticNet,4,default,0.338950,0.248143,-0.000154,0.001104,0.519774,0.342007,0.000000
5,Loughran-McDonald,XGBoost,4,default,0.340680,0.258086,-0.010390,-0.009120,0.474576,0.476246,0.126729
6,Loughran-McDonald,LightGBM,4,default,0.343281,0.254837,-0.025878,-0.024588,0.531073,0.536454,0.119498
7,No label,ExtraTrees,0,default,0.325107,0.245525,0.079870,0.081027,0.548023,0.532106,0.071687
8,No label,LinearRegression,0,default,0.330732,0.255733,0.047758,0.048956,0.519774,0.527210,0.115027
9,No label,Ridge,0,default,0.330790,0.255739,0.047423,0.048621,0.514124,0.521005,0.114598


TEST — refit train+validation


,labeling,model,n_label_features,best_params,RMSE,MAE,R2,R2_oos_vs_zero,DirAcc,F1,pred_std
0,Loughran-McDonald,ElasticNet,4,default,0.290304,0.218743,-0.004652,0.010005,0.584270,0.368794,0.000000
1,Loughran-McDonald,RandomForest,4,default,0.302180,0.217027,-0.088526,-0.072646,0.561798,0.479947,0.073492
2,Loughran-McDonald,ExtraTrees,4,default,0.306598,0.219238,-0.120594,-0.104245,0.573034,0.551668,0.090148
3,Loughran-McDonald,XGBoost,4,default,0.313520,0.231803,-0.171764,-0.154669,0.533708,0.484390,0.114200
4,Loughran-McDonald,LightGBM,4,default,0.315143,0.237290,-0.183923,-0.166651,0.522472,0.492006,0.124399
5,Loughran-McDonald,Ridge,4,default,0.339541,0.241689,-0.374334,-0.354284,0.516854,0.533436,0.173835
6,Loughran-McDonald,LinearRegression,4,default,0.340195,0.241854,-0.379638,-0.359510,0.516854,0.533436,0.175026
7,No label,ElasticNet,0,default,0.290304,0.218743,-0.004652,0.010005,0.584270,0.368794,0.000000
8,No label,RandomForest,0,default,0.301636,0.217892,-0.084612,-0.068788,0.561798,0.479947,0.073117
9,No label,ExtraTrees,0,default,0.307475,0.222105,-0.127009,-0.110567,0.539326,0.505469,0.084912


## Analisis pengaruh metode pelabelan

Bagian ini langsung menjawab pertanyaan penelitian: **label mana yang menambah informasi di atas embedding TF-IDF yang sama?**

`TF-IDF only` dipakai sebagai baseline. `Delta RMSE` yang negatif berarti penambahan label menghasilkan RMSE test yang lebih rendah daripada baseline TF-IDF untuk model yang sama.

In [32]:
# ===== DAMPAK LABELING: DELTA TERHADAP TF-IDF ONLY =====
base = (
    test_results[test_results['labeling'] == 'No label']
    [['model', 'RMSE', 'MAE', 'R2', 'DirAcc', 'F1']]
    .rename(columns={
        'RMSE': 'RMSE_base',
        'MAE': 'MAE_base',
        'R2': 'R2_base',
        'DirAcc': 'DirAcc_base',
        'F1': 'F1_base'
    })
)

impact = test_results.merge(base, on='model', how='left')
impact['Delta_RMSE'] = impact['RMSE'] - impact['RMSE_base']
impact['Delta_MAE'] = impact['MAE'] - impact['MAE_base']
impact['Delta_R2'] = impact['R2'] - impact['R2_base']
impact['Delta_DirAcc'] = impact['DirAcc'] - impact['DirAcc_base']
impact['Delta_F1'] = impact['F1'] - impact['F1_base']          # F1: naik = lebih baik (kebalikan RMSE/MAE)

impact = impact[impact['labeling'] != 'No label'].copy()

print('Dampak terhadap baseline TF-IDF only')
display(
    impact[[
        'labeling', 'model', 'RMSE', 'Delta_RMSE',
        'MAE', 'Delta_MAE', 'R2', 'Delta_R2',
        'DirAcc', 'Delta_DirAcc', 'F1', 'Delta_F1'
    ]]
    .sort_values(['model', 'Delta_RMSE'])
    .reset_index(drop=True)
)

# Rata-rata dampak lintas model sebagai ringkasan deskriptif,
# bukan sebagai pengganti hasil per-model.
impact_summary = (
    impact.groupby('labeling')
          .agg(
              mean_Delta_RMSE=('Delta_RMSE', 'mean'),
              mean_Delta_MAE=('Delta_MAE', 'mean'),
              mean_Delta_R2=('Delta_R2', 'mean'),
              mean_Delta_DirAcc=('Delta_DirAcc', 'mean'),
              mean_Delta_F1=('Delta_F1', 'mean')
          )
          .sort_values('mean_Delta_RMSE')
)

print('Ringkasan dampak rata-rata lintas model')
display(impact_summary)


Dampak terhadap baseline TF-IDF only


,labeling,model,RMSE,Delta_RMSE,MAE,Delta_MAE,R2,Delta_R2,DirAcc,Delta_DirAcc,F1,Delta_F1
0,VADER,ElasticNet,0.290304,0.000000,0.218743,0.000000,-0.004652,0.000000,0.584270,0.000000,0.368794,0.000000
1,Loughran-McDonald,ElasticNet,0.290304,0.000000,0.218743,0.000000,-0.004652,0.000000,0.584270,0.000000,0.368794,0.000000
2,VADER + LM,ElasticNet,0.290304,0.000000,0.218743,0.000000,-0.004652,0.000000,0.584270,0.000000,0.368794,0.000000
3,Loughran-McDonald,ExtraTrees,0.306598,-0.000876,0.219238,-0.002867,-0.120594,0.006415,0.573034,0.033708,0.551668,0.046199
4,VADER,ExtraTrees,0.308630,0.001155,0.221915,-0.000190,-0.135494,-0.008485,0.544944,0.005618,0.492621,-0.012847
5,VADER + LM,ExtraTrees,0.312123,0.004648,0.222164,0.000059,-0.161342,-0.034333,0.595506,0.056180,0.544057,0.038589
6,Loughran-McDonald,LightGBM,0.315143,-0.006493,0.237290,-0.001984,-0.183923,0.049285,0.522472,0.016854,0.492006,0.041862
7,VADER + LM,LightGBM,0.318208,-0.003427,0.240577,0.001303,-0.207067,0.026141,0.522472,0.016854,0.495475,0.045332
8,VADER,LightGBM,0.323708,0.002073,0.244415,0.005140,-0.249153,-0.015945,0.539326,0.033708,0.515317,0.065173
9,Loughran-McDonald,LinearRegression,0.340195,-0.000090,0.241854,0.000131,-0.379638,0.000728,0.516854,0.005618,0.533436,0.005102


Ringkasan dampak rata-rata lintas model


,mean_Delta_RMSE,mean_Delta_MAE,mean_Delta_R2,mean_Delta_DirAcc,mean_Delta_F1
labeling,,,,,
Loughran-McDonald,-0.001034,-0.001109,0.007857,0.013644,0.014115
VADER,-0.000149,0.000327,0.000909,0.020064,0.023314
VADER + LM,0.001929,0.002580,-0.014190,0.005618,0.009014


In [33]:
# ===== TABEL PIVOT UNTUK BAB HASIL =====
rmse_pivot = test_results.pivot(index='labeling', columns='model', values='RMSE')
mae_pivot = test_results.pivot(index='labeling', columns='model', values='MAE')
r2_pivot = test_results.pivot(index='labeling', columns='model', values='R2')
f1_pivot = test_results.pivot(index='labeling', columns='model', values='F1')

print('RMSE test')
display(rmse_pivot.round(6))

print('MAE test')
display(mae_pivot.round(6))

print('R² test')
display(r2_pivot.round(6))

print('F1 test (macro-F1 arah naik/turun)')
display(f1_pivot.round(6))


RMSE test


model,ElasticNet,ExtraTrees,LightGBM,LinearRegression,RandomForest,Ridge,XGBoost
labeling,,,,,,,
Loughran-McDonald,0.290304,0.306598,0.315143,0.340195,0.302180,0.339541,0.313520
No label,0.290304,0.307475,0.321635,0.340285,0.301636,0.339624,0.313762
VADER,0.290304,0.308630,0.323708,0.340966,0.302421,0.340307,0.307345
VADER + LM,0.290304,0.312123,0.318208,0.340323,0.307431,0.339670,0.320165


MAE test


model,ElasticNet,ExtraTrees,LightGBM,LinearRegression,RandomForest,Ridge,XGBoost
labeling,,,,,,,
Loughran-McDonald,0.218743,0.219238,0.237290,0.241854,0.217027,0.241689,0.231803
No label,0.218743,0.222105,0.239274,0.241724,0.217892,0.241566,0.234106
VADER,0.218743,0.221915,0.244415,0.242765,0.218589,0.242604,0.228669
VADER + LM,0.218743,0.222164,0.240577,0.242188,0.220597,0.242037,0.247166


R² test


model,ElasticNet,ExtraTrees,LightGBM,LinearRegression,RandomForest,Ridge,XGBoost
labeling,,,,,,,
Loughran-McDonald,-0.004652,-0.120594,-0.183923,-0.379638,-0.088526,-0.374334,-0.171764
No label,-0.004652,-0.127009,-0.233208,-0.380366,-0.084612,-0.375010,-0.173573
VADER,-0.004652,-0.135494,-0.249153,-0.385896,-0.090265,-0.380542,-0.126061
VADER + LM,-0.004652,-0.161342,-0.207067,-0.380671,-0.126688,-0.375380,-0.221961


F1 test (macro-F1 arah naik/turun)


model,ElasticNet,ExtraTrees,LightGBM,LinearRegression,RandomForest,Ridge,XGBoost
labeling,,,,,,,
Loughran-McDonald,0.368794,0.551668,0.492006,0.533436,0.479947,0.533436,0.484390
No label,0.368794,0.505469,0.450144,0.528334,0.479947,0.528334,0.483852
VADER,0.368794,0.492621,0.515317,0.527126,0.470238,0.538521,0.595455
VADER + LM,0.368794,0.544057,0.495475,0.533436,0.490919,0.533436,0.441856


## Catatan interpretasi

- Semua eksperimen memakai **embedding TF-IDF → SVD yang sama**; yang berubah hanya fitur label.
- `No label` = komponen SVD + fitur time-series/volume berita tanpa sinyal VADER atau Loughran-McDonald.
- `VADER` dan `Loughran-McDonald` diuji sendiri-sendiri agar efek masing-masing dapat diisolasi.
- `VADER + LM` menguji apakah kombinasi dua metode pelabelan memberi informasi tambahan.
- Untuk menyatakan suatu metode berpengaruh pada performa, lihat perubahan **RMSE/MAE/R²** terhadap `No label` pada **model yang sama**.
- Hanya **satu model dengan parameter default** yang dipakai, jadi kesimpulan tentang label tidak bercampur dengan efek tuning. Konsekuensinya, hasil ini belum menunjukkan apakah model lain atau hyperparameter berbeda akan memberi kesimpulan yang sama.
- `DirAcc` dan `F1` hanya metrik tambahan untuk melihat apakah arah prediksi return ikut membaik; target utama tetap regresi return kontinu.
- `F1` adalah macro-F1 pada arah naik (return > 0) vs turun/tetap (return <= 0). Nilainya perlu dibaca bersama proporsi hari naik di tiap split (dicetak pada cell split): prediksi yang hampir selalu "naik" atau selalu "turun" akan menghasilkan macro-F1 rendah.